# Deploy and call r-1 mini on Amazon SageMaker

This notebook uses `boto3` to deploy the Reducto r-1 mini model package from AWS Marketplace, parse pages in real time and in batch, and delete the resources.

r-1 mini takes one page image and returns the layout blocks of the page. Each block has a `type`, a `bbox` on the 0 to 1 page scale, and `content`. See [`../README.md`](../README.md) for the full contract.

Steps:

1. Set up
2. Create the model
3. Deploy a real-time endpoint
4. Send the page-parse body
5. Send the chat-completions body
6. Handle a page that r-1 mini cannot process
7. Delete the endpoint
8. Run a batch transform job
9. Delete the model and the batch files

Prerequisites:

- A subscription to r-1 mini through Amazon SageMaker on AWS Marketplace, and the ARN of model package version 3 or later (**AWS Marketplace > Manage subscriptions > Configure**).
- A SageMaker execution role. For batch transform, the role must read and write the S3 prefix below.
- Quota for one `ml.g6e.2xlarge` for endpoint usage and one for transform job usage.
- Python 3.9 or later.

AWS bills the endpoint instance while the endpoint exists. If a cell fails after step 3, run steps 7 and 9 to delete the resources.

## 1. Set up

Replace the placeholders with your own values. The cell prints the names of the resources that this notebook creates. If the kernel restarts, set `run_id` to the printed value, so that the clean-up steps find the resources.

In [ ]:
%pip install -q "boto3>=1.34"

In [ ]:
import base64
import json
import time
from pathlib import Path
from typing import Optional

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

region = "<region>"  # The region of your subscription, for example "us-east-2".
model_package_arn = "<model-package-arn>"  # From Manage subscriptions > Configure.
role_arn = "<execution-role-arn>"  # Your SageMaker execution role.
bucket = "<bucket>"  # An S3 bucket in the same region, for batch transform.

settings = {
    "region": region,
    "model_package_arn": model_package_arn,
    "role_arn": role_arn,
    "bucket": bucket,
}
for name, value in settings.items():
    if value.startswith("<"):
        raise ValueError(f"Set {name} before you continue.")

run_id = time.strftime("%Y%m%d-%H%M%S")  # After a kernel restart, set this to the printed value.
instance_type = "ml.g6e.2xlarge"
model_name = f"r-1-mini-{run_id}"
endpoint_name = f"r-1-mini-{run_id}"
transform_job_name = f"r-1-mini-batch-{run_id}"
prefix = f"r-1-mini-batch/{run_id}"
print("run_id:", run_id)
print("model:", model_name)
print("endpoint:", endpoint_name)
print("transform job:", transform_job_name)
print("S3 prefix:", f"s3://{bucket}/{prefix}/")

sagemaker = boto3.client("sagemaker", region_name=region)
# The endpoint answers within 60 seconds. A longer read timeout prevents a second send.
runtime = boto3.client("sagemaker-runtime", region_name=region, config=Config(read_timeout=70))
s3 = boto3.client("s3", region_name=region)

# Helpers for the clean-up steps.
def delete_if_exists(delete, **name) -> None:
    """Call a SageMaker delete function, and accept a resource that does not exist."""
    try:
        delete(**name)
    except ClientError as error:
        if "Could not find" not in error.response["Error"].get("Message", ""):
            raise
        print("Not found:", name)


def list_keys(key_prefix: str) -> list[str]:
    pages = s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=key_prefix)
    return [obj["Key"] for page in pages for obj in page.get("Contents", [])]


# Works whether Jupyter starts in this folder or at the repository root.
samples = next((p for p in (Path("../samples"), Path("r-1-mini/samples")) if p.is_dir()), None)
if samples is None:
    raise FileNotFoundError("Start Jupyter in this folder or at the repository root.")

## 2. Create the model

A model from an AWS Marketplace model package must run with network isolation.

In [ ]:
sagemaker.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role_arn,
    PrimaryContainer={"ModelPackageName": model_package_arn},
    EnableNetworkIsolation=True,
)

## 3. Deploy a real-time endpoint

The endpoint can take 10 to 20 minutes to start.

In [ ]:
sagemaker.create_endpoint_config(
    EndpointConfigName=endpoint_name,
    ProductionVariants=[
        {
            "VariantName": "primary",
            "ModelName": model_name,
            "InstanceType": instance_type,
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": 3600,
        }
    ],
)
sagemaker.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_name)
sagemaker.get_waiter("endpoint_in_service").wait(
    EndpointName=endpoint_name, WaiterConfig={"Delay": 30, "MaxAttempts": 80}
)
print(sagemaker.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"])

## 4. Send the page-parse body

The page-parse body is `{"image_b64": "<base64 PNG or JPEG>"}`. The answer is `{"blocks": [...], "usage": {...}}`.

This cell sends the sample request [`invoice.request.json`](../samples/invoice.request.json).

In [ ]:
response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=(samples / "invoice.request.json").read_bytes(),
)
result = json.loads(response["Body"].read())
if "blocks" not in result:
    raise RuntimeError("The answer has no blocks. Deploy model package version 3 or later.")
print(result["usage"])
for block in result["blocks"]:
    print(block["type"], block["bbox"])
    print(block["content"])

To send your own page, base64-encode a PNG or JPEG. The `bbox` values are fractions of the page width and height. Multiply them by the image size to get pixels.

In [ ]:
PNG_SIGNATURE = b"\x89PNG\r\n\x1a\n"


def page_body(image_path: Path) -> str:
    return json.dumps({"image_b64": base64.b64encode(image_path.read_bytes()).decode()})


def png_size(image_path: Path) -> tuple[int, int]:
    """Read the width and height from a PNG file header."""
    header = image_path.read_bytes()[:24]
    if not header.startswith(PNG_SIGNATURE):
        raise ValueError(f"{image_path} is not a PNG. Use an image library for other types.")
    return int.from_bytes(header[16:20], "big"), int.from_bytes(header[20:24], "big")


page = samples / "text.png"
response = runtime.invoke_endpoint(
    EndpointName=endpoint_name, ContentType="application/json", Body=page_body(page)
)
width, height = png_size(page)
for block in json.loads(response["Body"].read())["blocks"]:
    box = block["bbox"]
    pixels = [
        round(box["left"] * width),
        round(box["top"] * height),
        round(box["width"] * width),
        round(box["height"] * height),
    ]
    print(f"{block['type']:<10} x, y, w, h = {pixels}  {block['content']}")

## 5. Send the chat-completions body

The chat-completions body is an OpenAI chat-completions request with one `user` message and one `image_url` part. The image must be a `data:` URL of a PNG or JPEG.

The answer is a `chat.completion` object. `choices[0].message.content` holds the same `{"blocks": [...]}` object as a JSON string.

In [ ]:
def image_mime_type(data: bytes) -> str:
    if data.startswith(PNG_SIGNATURE):
        return "image/png"
    if data.startswith(b"\xff\xd8\xff"):
        return "image/jpeg"
    raise ValueError("r-1 mini accepts PNG or JPEG images only")


def chat_body(image_path: Path) -> str:
    data = image_path.read_bytes()
    data_url = f"data:{image_mime_type(data)};base64,{base64.b64encode(data).decode()}"
    return json.dumps(
        {
            "model": "r-1-mini",
            "messages": [
                {
                    "role": "user",
                    "content": [{"type": "image_url", "image_url": {"url": data_url}}],
                }
            ],
        }
    )


response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=chat_body(samples / "invoice.png"),
)
completion = json.loads(response["Body"].read())
print(completion["model"], completion["choices"][0]["finish_reason"], completion["usage"])
for block in json.loads(completion["choices"][0]["message"]["content"])["blocks"]:
    print(block["type"], block["content"])

## 6. Handle a page that r-1 mini cannot process

When r-1 mini cannot process a page, the endpoint returns HTTP 422 with `{"detail": "input could not be processed"}`. SageMaker raises it as a 424 `ModelError` with `OriginalStatusCode` 422.

Do not retry this page, because a retry gives the same result. A 422 with the detail `invalid request` means that the request body is not valid. The function below raises every other error. See the error table in [`../README.md`](../README.md#errors) for the errors that you can retry.

In [ ]:
UNPROCESSABLE = "input could not be processed"


def parse_page(body: str) -> Optional[list[dict]]:
    """Return the blocks of one page, or None when r-1 mini cannot process the page."""
    try:
        response = runtime.invoke_endpoint(
            EndpointName=endpoint_name, ContentType="application/json", Body=body
        )
    except runtime.exceptions.ModelError as error:
        status = error.response.get("OriginalStatusCode")
        message = error.response.get("OriginalMessage") or ""
        if status == 422 and UNPROCESSABLE in message:
            return None  # Do not retry: a retry gives the same result.
        raise
    return json.loads(response["Body"].read())["blocks"]


for name in ["invoice", "text", "rotated"]:
    blocks = parse_page(page_body(samples / f"{name}.png"))
    print(name, "could not be processed" if blocks is None else f"{len(blocks)} blocks")

## 7. Delete the endpoint

AWS bills the endpoint instance while the endpoint exists. The batch transform job in step 8 starts its own instance, so it does not need the endpoint.

In [ ]:
delete_if_exists(sagemaker.delete_endpoint, EndpointName=endpoint_name)
sagemaker.get_waiter("endpoint_deleted").wait(EndpointName=endpoint_name)
delete_if_exists(sagemaker.delete_endpoint_config, EndpointConfigName=endpoint_name)

## 8. Run a batch transform job

Each S3 object holds one JSON request body. Both bodies work. The job writes one `<input>.out` object for each input, with the same JSON as the real-time response.

Use these settings:

| Setting | Value |
|---|---|
| `ContentType` | `application/json` |
| `SplitType` | `None` |
| `BatchStrategy` | `SingleRecord` |
| `MaxPayloadInMB` | `6` |

When r-1 mini cannot process a page, the `.out` object of that page holds `{"detail": "input could not be processed"}`, and the job continues. Any other error on a record, for example an image that is not a readable PNG or JPEG, fails the whole job. Check your images before you upload them.

In [ ]:
inputs = {name: page_body(samples / f"{name}.png") for name in ["invoice", "text", "rotated"]}
inputs["invoice-chat"] = (samples / "invoice.chat.request.json").read_text()
for name, body in inputs.items():
    s3.put_object(
        Bucket=bucket,
        Key=f"{prefix}/input/{name}.json",
        Body=body.encode(),
        ContentType="application/json",
    )

sagemaker.create_transform_job(
    TransformJobName=transform_job_name,
    ModelName=model_name,
    BatchStrategy="SingleRecord",
    MaxPayloadInMB=6,
    MaxConcurrentTransforms=8,
    TransformInput={
        "DataSource": {
            "S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": f"s3://{bucket}/{prefix}/input/"}
        },
        "ContentType": "application/json",
        "SplitType": "None",
    },
    TransformOutput={
        "S3OutputPath": f"s3://{bucket}/{prefix}/output/",
        "Accept": "application/json",
        "AssembleWith": "None",
    },
    TransformResources={"InstanceType": instance_type, "InstanceCount": 1},
)
sagemaker.get_waiter("transform_job_completed_or_stopped").wait(
    TransformJobName=transform_job_name, WaiterConfig={"Delay": 60, "MaxAttempts": 120}
)
job = sagemaker.describe_transform_job(TransformJobName=transform_job_name)
if job["TransformJobStatus"] != "Completed":
    status, reason = job["TransformJobStatus"], job.get("FailureReason")
    raise RuntimeError(f"The transform job is {status}: {reason}")
print(job["TransformJobStatus"])

In [ ]:
outputs = list_keys(f"{prefix}/output/")
missing = {name for name in inputs} - {Path(key).name.removesuffix(".json.out") for key in outputs}
if missing:
    print("No output for:", sorted(missing))

for key in outputs:
    output = json.loads(s3.get_object(Bucket=bucket, Key=key)["Body"].read())
    if "detail" in output:
        print(f"== {key}: {output['detail']}")
        continue
    if "choices" in output:
        output = json.loads(output["choices"][0]["message"]["content"])
    print(f"== {key}: {len(output['blocks'])} blocks")
    for block in output["blocks"]:
        print(" ", block["type"], block["content"][:80])

## 9. Delete the model and the batch files

Delete the model and the S3 objects of this run. The bucket stays. If the transform job still runs, the cell stops it first.

In [ ]:
try:
    job = sagemaker.describe_transform_job(TransformJobName=transform_job_name)
    status = job["TransformJobStatus"]
except ClientError as error:
    if "Could not find" not in error.response["Error"].get("Message", ""):
        raise
    status = None
if status == "InProgress":
    sagemaker.stop_transform_job(TransformJobName=transform_job_name)
    sagemaker.get_waiter("transform_job_completed_or_stopped").wait(
        TransformJobName=transform_job_name
    )

delete_if_exists(sagemaker.delete_model, ModelName=model_name)

keys = list_keys(f"{prefix}/")
for start in range(0, len(keys), 1000):
    batch = [{"Key": key} for key in keys[start : start + 1000]]
    result = s3.delete_objects(Bucket=bucket, Delete={"Objects": batch})
    if result.get("Errors"):
        raise RuntimeError(f"Some S3 objects were not deleted: {result['Errors']}")
print(f"Deleted {len(keys)} S3 objects.")

To unsubscribe, delete all endpoints and models from the model package first. Then open **AWS Marketplace > Manage subscriptions** in the console and cancel the r-1 mini subscription.